# MCP: métodos estándar con Everything MCP Server

Este notebook usa como ejemplo principal **Everything MCP Server**, un servidor de referencia pensado específicamente para probar muchas capacidades del protocolo MCP.

Lo ejecutaremos localmente por **stdio** mediante:

`npx -y @modelcontextprotocol/server-everything`

Como comparación, al final dejamos también **OpenAI Docs MCP** como ejemplo de servidor remoto más limitado.

## Flujo

`initialize → capabilities → tools/resources/prompts → ejecución → timeline → análisis seguridad`

## 0. Qué esperamos del Everything MCP

Everything MCP declara varias capacidades:

- Tools
- Prompts
- Resources
- Logging
- Tasks

Además, resources soporta suscripción y cambios de lista.

Esto lo hace más útil que un MCP limitado para estudiar el protocolo completo.

In [1]:
%pip install -q "mcp>=2,<3"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\p.ledesma.jimenez\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [4]:
from __future__ import annotations

import json
import shutil
import sys
from datetime import datetime
from pathlib import Path
from mcp import Client, StdioServerParameters, stdio_client
from mcp.types import TextContent

print("Python:", sys.version.split()[0])
print("npx encontrado:", shutil.which("npx"))

Python: 3.14.6
npx encontrado: C:\Program Files\nodejs\npx.CMD


# 1. Arrancar Everything MCP por stdio

El cliente lanza Everything MCP como proceso local.

En Windows usamos:

`cmd /c npx -y @modelcontextprotocol/server-everything`

### Nota para Jupyter en Windows

Jupyter sustituye `sys.stderr` por un stream propio que puede no tener `fileno()`.

El SDK MCP necesita un stream de sistema válido al crear el subproceso, así que este notebook redirige el `stderr` del servidor a:

`everything_mcp_stderr.log`

El tráfico MCP sigue viajando normalmente por **stdin/stdout**; solo los mensajes de error/log del proceso hijo van al fichero.

In [5]:
import os

if os.name == "nt":
    EVERYTHING_SERVER = StdioServerParameters(
        command="cmd",
        args=["/c", "npx", "-y", "@modelcontextprotocol/server-everything"],
    )
else:
    EVERYTHING_SERVER = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-everything"],
    )

print(EVERYTHING_SERVER)

# --- Jupyter + Windows compatibility ---
# Jupyter's sys.stderr may not expose fileno(), which can break subprocess
# creation on Windows. Redirect the child process stderr to a real file.
EVERYTHING_LOG_PATH = Path("everything_mcp_stderr.log").resolve()
EVERYTHING_ERRLOG = open(
    EVERYTHING_LOG_PATH,
    "a",
    encoding="utf-8",
    buffering=1,
)

def everything_transport():
    """Fresh stdio transport with child stderr redirected to a real file."""
    return stdio_client(
        EVERYTHING_SERVER,
        errlog=EVERYTHING_ERRLOG,
    )

print("MCP stderr log:", EVERYTHING_LOG_PATH)


command='cmd' args=['/c', 'npx', '-y', '@modelcontextprotocol/server-everything'] env=None cwd=None encoding='utf-8' encoding_error_handler='strict'
MCP stderr log: C:\Users\p.ledesma.jimenez\scripts\everything_mcp_stderr.log


### Comprobación rápida

Si el servidor no arranca, revisamos su `stderr` sin interferir con el canal MCP.

In [6]:
print("Log de Everything MCP:", EVERYTHING_LOG_PATH)

if EVERYTHING_LOG_PATH.exists() and EVERYTHING_LOG_PATH.stat().st_size:
    print(EVERYTHING_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-3000:])
else:
    print("El log está vacío por ahora.")

Log de Everything MCP: C:\Users\p.ledesma.jimenez\scripts\everything_mcp_stderr.log
El log está vacío por ahora.


# 2. `initialize`

Al conectar se negocian versión y capacidades.

Aquí interesa especialmente observar `server_info` y `server_capabilities`.

In [7]:
async with Client(everything_transport()) as client:
    print("protocol_version:", client.protocol_version)
    print("server_info:", client.server_info)
    print("server_capabilities:")
    print(client.server_capabilities)

protocol_version: 2025-11-25
server_info: name='mcp-servers/everything' title='Everything Reference Server' version='2.0.0' description=None website_url=None icons=None
server_capabilities:
experimental=None logging=LoggingCapability() prompts=PromptsCapability(list_changed=True) resources=ResourcesCapability(subscribe=True, list_changed=True) tools=ToolsCapability(list_changed=True) completions=CompletionsCapability() extensions=None tasks=ServerTasksCapability(list=TasksListCapability(), cancel=TasksCancelCapability(), requests=ServerTasksRequestsCapability(tools=TasksToolsCapability(call=TasksCallCapability())))


## Qué debería aparecer

A diferencia de OpenAI Docs MCP, aquí esperamos ver varias capabilities activas, por ejemplo:

- `tools`
- `prompts`
- `resources`
- `logging`
- `tasks`

Esto permite comprobar qué información de identificación y capacidades aparece ya en el `InitializeResult`.

# 3. `tools/list`

Descubrimos las tools disponibles.

Resultado conceptual: `ListToolsResult`.

In [8]:
async with Client(everything_transport()) as client:
    result = await client.list_tools()

    print("Tipo:", type(result).__name__)
    print("Número de tools:", len(result.tools))

    for tool in result.tools:
        print("\n---", tool.name, "---")
        print(tool.description)
        print(json.dumps(tool.input_schema, indent=2, ensure_ascii=False))

Tipo: ListToolsResult
Número de tools: 13

--- echo ---
Echoes back the input string
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "type": "object",
  "properties": {
    "message": {
      "type": "string",
      "description": "Message to echo"
    }
  },
  "required": [
    "message"
  ]
}

--- get-annotated-message ---
Demonstrates how annotations can be used to provide metadata about content.
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "type": "object",
  "properties": {
    "messageType": {
      "type": "string",
      "enum": [
        "error",
        "success",
        "debug"
      ],
      "description": "Type of message to demonstrate different annotation patterns"
    },
    "includeImage": {
      "default": false,
      "description": "Whether to include an example image",
      "type": "boolean"
    }
  },
  "required": [
    "messageType"
  ]
}

--- get-env ---
Returns all environment variables, helpful for debugging MCP server configurat

# 4. `resources/list`

Everything MCP expone resources, así que aquí sí podemos probar esta superficie realmente.

In [9]:
async with Client(everything_transport()) as client:
    result = await client.list_resources()

    print("Tipo:", type(result).__name__)
    print("Número de resources:", len(result.resources))

    for resource in result.resources:
        print("\n--- RESOURCE ---")
        print("name:", resource.name)
        print("uri:", resource.uri)
        print("mime_type:", getattr(resource, "mime_type", None))

Tipo: ListResourcesResult
Número de resources: 7

--- RESOURCE ---
name: architecture.md
uri: demo://resource/static/document/architecture.md
mime_type: text/markdown

--- RESOURCE ---
name: extension.md
uri: demo://resource/static/document/extension.md
mime_type: text/markdown

--- RESOURCE ---
name: features.md
uri: demo://resource/static/document/features.md
mime_type: text/markdown

--- RESOURCE ---
name: how-it-works.md
uri: demo://resource/static/document/how-it-works.md
mime_type: text/markdown

--- RESOURCE ---
name: instructions.md
uri: demo://resource/static/document/instructions.md
mime_type: text/markdown

--- RESOURCE ---
name: startup.md
uri: demo://resource/static/document/startup.md
mime_type: text/markdown

--- RESOURCE ---
name: structure.md
uri: demo://resource/static/document/structure.md
mime_type: text/markdown


# 5. `resources/templates/list`

Descubre plantillas de recursos parametrizables.

In [10]:
async with Client(everything_transport()) as client:
    result = await client.list_resource_templates()

    print("Tipo:", type(result).__name__)
    print("Templates:", len(result.resource_templates))

    for template in result.resource_templates:
        print("\n--- TEMPLATE ---")
        print("name:", template.name)
        print("uri_template:", template.uri_template)

Tipo: ListResourceTemplatesResult
Templates: 2

--- TEMPLATE ---
name: Dynamic Text Resource
uri_template: demo://resource/dynamic/text/{resourceId}

--- TEMPLATE ---
name: Dynamic Blob Resource
uri_template: demo://resource/dynamic/blob/{resourceId}


# 6. `resources/read`

Leemos un recurso real.

Request / Result:

`ReadResourceRequest → ReadResourceResult`

In [11]:
async with Client(everything_transport()) as client:
    resources = await client.list_resources()

    if not resources.resources:
        print("No hay resources.")
    else:
        resource = resources.resources[0]

        print("Leyendo:", resource.uri)
        result = await client.read_resource(resource.uri)

        print("Tipo:", type(result).__name__)
        print(result)

Leyendo: demo://resource/static/document/architecture.md
Tipo: ReadResourceResult
meta=None ttl_ms=0 cache_scope='private' contents=[TextResourceContents(uri='demo://resource/static/document/architecture.md', mime_type='text/markdown', meta=None, text='# Everything Server – Architecture\n\n**Architecture\n| [Project Structure](structure.md)\n| [Startup Process](startup.md)\n| [Server Features](features.md)\n| [Extension Points](extension.md)\n| [How It Works](how-it-works.md)**\n\nThis documentation summarizes the current layout and runtime architecture of the `src/everything` package.\nIt explains how the server starts, how transports are wired, where tools, prompts, and resources are registered, and how to extend the system.\n\n## High‑level Overview\n\n### Purpose\n\nA minimal, modular MCP server showcasing core Model Context Protocol features. It exposes simple tools, prompts, and resources, and can be run over multiple transports (STDIO, SSE, and Streamable HTTP).\n\n### Design\n\

# 7. `prompts/list`

Descubrimos prompts registrados en el servidor.

In [12]:
async with Client(everything_transport()) as client:
    result = await client.list_prompts()

    print("Tipo:", type(result).__name__)
    print("Número de prompts:", len(result.prompts))

    for prompt in result.prompts:
        print("\n--- PROMPT ---")
        print("name:", prompt.name)
        print("description:", prompt.description)
        print("arguments:", prompt.arguments)

Tipo: ListPromptsResult
Número de prompts: 4

--- PROMPT ---
name: simple-prompt
description: A prompt with no arguments
arguments: None

--- PROMPT ---
name: args-prompt
description: A prompt with two arguments, one required and one optional
arguments: [PromptArgument(name='city', title=None, description='Name of the city', required=True), PromptArgument(name='state', title=None, description=None, required=False)]

--- PROMPT ---
name: completable-prompt
description: First argument choice narrows values for second argument.
arguments: [PromptArgument(name='department', title=None, description='Choose the department.', required=True), PromptArgument(name='name', title=None, description='Choose a team member to lead the selected department.', required=True)]

--- PROMPT ---
name: resource-prompt
description: A prompt that includes an embedded resource reference
arguments: [PromptArgument(name='resourceType', title=None, description='Type of resource to fetch', required=True), PromptArgu

# 8. `prompts/get`

Recuperamos un prompt concreto.

Si el primer prompt necesita argumentos obligatorios, los mostramos antes de intentar la llamada.

In [13]:
async with Client(everything_transport()) as client:
    prompts = await client.list_prompts()

    if not prompts.prompts:
        print("No hay prompts.")
    else:
        prompt = prompts.prompts[0]

        print("Prompt:", prompt.name)
        print("Arguments:", prompt.arguments)

        required_args = [
            arg.name
            for arg in (prompt.arguments or [])
            if getattr(arg, "required", False)
        ]

        if required_args:
            print("Este prompt requiere argumentos:", required_args)
            print("No lo ejecutamos automáticamente.")
        else:
            result = await client.get_prompt(prompt.name)
            print("Tipo:", type(result).__name__)
            print(result)

Prompt: simple-prompt
Arguments: None
Tipo: GetPromptResult
meta=None description=None messages=[PromptMessage(role='user', content=TextContent(type='text', text='This is a simple prompt without arguments.', annotations=None, meta=None))] result_type='complete'


# 9. `tools/call`

Ejecutamos una tool real.

Intentaremos usar `echo`, una de las tools sencillas del Everything MCP.

In [14]:
async with Client(everything_transport()) as client:
    tools = await client.list_tools()
    by_name = {tool.name: tool for tool in tools.tools}

    print("Tools disponibles:", list(by_name))

    if "echo" in by_name:
        result = await client.call_tool(
            "echo",
            {"message": "Hola desde Jupyter"}
        )

        print("Tipo:", type(result).__name__)
        print("is_error:", result.is_error)

        for block in result.content:
            print(block.text if isinstance(block, TextContent) else block)
    else:
        print("No se encontró la tool echo.")

Tools disponibles: ['echo', 'get-annotated-message', 'get-env', 'get-resource-links', 'get-resource-reference', 'get-structured-content', 'get-sum', 'get-tiny-image', 'gzip-file-as-resource', 'toggle-simulated-logging', 'toggle-subscriber-updates', 'trigger-long-running-operation', 'simulate-research-query']
Tipo: CallToolResult
is_error: False
Echo: Hola desde Jupyter


# 10. Logging

Everything MCP declara la capability `logging`.

Esto significa que el servidor puede enviar mensajes de log al cliente siguiendo el protocolo MCP.

Aquí no forzamos todavía un flujo completo de logging; primero comprobamos simplemente que la capability está declarada.

In [15]:
async with Client(everything_transport()) as client:
    caps = client.server_capabilities

    print("logging:", caps.logging)
    print("tools:", caps.tools)
    print("prompts:", caps.prompts)
    print("resources:", caps.resources)
    print("tasks:", caps.tasks)

logging: 
tools: list_changed=True
prompts: list_changed=True
resources: subscribe=True list_changed=True
tasks: list=TasksListCapability() cancel=TasksCancelCapability() requests=ServerTasksRequestsCapability(tools=TasksToolsCapability(call=TasksCallCapability()))


# 11. Timeline de una sesión MCP

Construimos una secuencia simple de eventos para visualizar que una sesión no es un único registro.

In [16]:
timeline = []

def log_event(direction, method, detail=""):
    timeline.append({
        "timestamp": datetime.now().isoformat(timespec="milliseconds"),
        "direction": direction,
        "method": method,
        "detail": detail,
    })

async with Client(everything_transport()) as client:
    log_event("client → server", "initialize", "inicio sesión")
    log_event("server → client", "InitializeResult", str(client.server_info))

    tools = await client.list_tools()
    log_event("client → server", "tools/list", "discovery")
    log_event("server → client", "ListToolsResult", f"{len(tools.tools)} tools")

    resources = await client.list_resources()
    log_event("client → server", "resources/list", "discovery")
    log_event("server → client", "ListResourcesResult", f"{len(resources.resources)} resources")

    prompts = await client.list_prompts()
    log_event("client → server", "prompts/list", "discovery")
    log_event("server → client", "ListPromptsResult", f"{len(prompts.prompts)} prompts")

    if any(t.name == "echo" for t in tools.tools):
        log_event("client → server", "tools/call", "echo")
        result = await client.call_tool("echo", {"message": "timeline test"})
        log_event("server → client", "CallToolResult", f"is_error={result.is_error}")

for event in timeline:
    print(
        event["timestamp"],
        "|",
        event["direction"].ljust(15),
        "|",
        event["method"].ljust(22),
        "|",
        event["detail"]
    )

2026-10-02T12:34:05.320 | client → server | initialize             | inicio sesión
2026-10-02T12:34:05.321 | server → client | InitializeResult       | name='mcp-servers/everything' title='Everything Reference Server' version='2.0.0' description=None website_url=None icons=None
2026-10-02T12:34:05.327 | client → server | tools/list             | discovery
2026-10-02T12:34:05.327 | server → client | ListToolsResult        | 13 tools
2026-10-02T12:34:05.329 | client → server | resources/list         | discovery
2026-10-02T12:34:05.329 | server → client | ListResourcesResult    | 7 resources
2026-10-02T12:34:05.332 | client → server | prompts/list           | discovery
2026-10-02T12:34:05.332 | server → client | ListPromptsResult      | 4 prompts
2026-10-02T12:34:05.332 | client → server | tools/call             | echo
2026-10-02T12:34:05.336 | server → client | CallToolResult         | is_error=False


# 12. Discovery vs interacción efectiva

Para el análisis de seguridad podemos clasificar los métodos de forma simple:

In [17]:
MCP_OPERATION_CLASSIFICATION = {
    "initialize": "Discovery",
    "tools/list": "Discovery",
    "resources/list": "Discovery",
    "resources/templates/list": "Discovery",
    "prompts/list": "Discovery",

    "prompts/get": "Effective interaction",
    "resources/read": "Effective interaction",
    "tools/call": "Execution",
}

for method, category in MCP_OPERATION_CLASSIFICATION.items():
    print(f"{method:28} -> {category}")

initialize                   -> Discovery
tools/list                   -> Discovery
resources/list               -> Discovery
resources/templates/list     -> Discovery
prompts/list                 -> Discovery
prompts/get                  -> Effective interaction
resources/read               -> Effective interaction
tools/call                   -> Execution


Una política podría conceptualmente permitir discovery y aplicar más restricciones cuando empiezan:

- lectura de datos;
- recuperación de prompts;
- ejecución de tools.

# 13. Qué podemos validar con este servidor

| Prueba | Pregunta |
|---|---|
| `initialize` | ¿Qué identidad y capabilities aparecen al conectar? |
| `tools/list` | ¿Qué herramientas puede descubrir el cliente? |
| `resources/list` | ¿Qué datos expone el servidor? |
| `resources/read` | ¿Cuándo comienza acceso real a datos? |
| `prompts/list` | ¿Qué prompts son visibles? |
| `prompts/get` | ¿Cuándo comienza una interacción efectiva? |
| `tools/call` | ¿Cuándo se ejecuta una acción? |
| Timeline | ¿Hay que correlacionar varios eventos? |
| Capabilities | ¿Qué superficies puede usar una política de seguridad? |

# 14. Comparación: OpenAI Docs MCP remoto

Ahora conectamos a un MCP real remoto para comparar sus capabilities con Everything MCP.

In [18]:
OPENAI_DOCS_MCP = "https://developers.openai.com/mcp"

async with Client(OPENAI_DOCS_MCP) as client:
    print("server_info:", client.server_info)
    print("server_capabilities:")
    print(client.server_capabilities)

server_info: name='openai-docs-mcp' title=None version='1.0.0' description=None website_url=None icons=None
server_capabilities:
experimental=None logging=None prompts=None resources=None tools=ToolsCapability(list_changed=True) completions=None extensions=None tasks=None


## Comparación esperada

**Everything MCP**
- pensado para pruebas;
- muchas capabilities;
- ideal para estudiar MCP.

**OpenAI Docs MCP**
- servidor real orientado a una función concreta;
- normalmente expone menos superficies;
- útil para comparar laboratorio vs caso real.

El transporte cambia (`stdio` vs Streamable HTTP), pero los métodos MCP siguen siendo los mismos.